# 01 - Data Exploration

**Project:** NLP Text Classification Pipeline for Public-Sector Documents

**Dataset:** Synthetic public-sector documents (FOIA requests, 311 service requests, policy notices, budget documents) based on real-world patterns from Federal Register, DC Open Data, and MuckRock.

**Objective:** Explore document categories, text length distributions, and vocabulary patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded.")

In [ ]:
DATA_DIR = '../data'

df = pd.read_csv(f'{DATA_DIR}/documents.csv')
df['created_date'] = pd.to_datetime(df['created_date'])

print(f"Documents: {len(df):,}")
print(f"Categories: {df['category'].unique()}")
print(f"Date range: {df['created_date'].min()} to {df['created_date'].max()}")

print(f"\nCategory distribution:")
print(df['category'].value_counts())

print(f"\nPriority distribution:")
print(df['priority'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Category distribution
df['category'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Document Category Distribution')
axes[0,0].tick_params(axis='x', rotation=45)

# Word count by category
sns.boxplot(data=df, x='category', y='word_count', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Word Count by Category')
axes[0,1].tick_params(axis='x', rotation=45)

# Documents over time
df['created_date'].dt.to_period('M').value_counts().sort_index().plot(
    ax=axes[1,0], color='coral'
)
axes[1,0].set_title('Documents Over Time')

# Attachment rate by category
attach_rate = df.groupby('category')['has_attachment'].mean()
attach_rate.plot(kind='bar', ax=axes[1,1], color='seagreen')
axes[1,1].set_title('Attachment Rate by Category')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/nlp_01_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/nlp_01_overview.png")

## Text Preprocessing

In [ ]:
import nltk
from sklearn.feature_extraction.text import CountVectorizer

# Download NLTK data if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

def preprocess_text(text: str) -> str:
    """Basic text preprocessing."""
    # Lowercase
    text = text.lower()
    # Remove special chars but keep spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

df['text_clean'] = df['text'].apply(preprocess_text)

# Sample documents
print("=== SAMPLE DOCUMENTS ===")
for cat in df['category'].unique():
    sample = df[df['category'] == cat].iloc[0]
    print(f"\n{cat}:")
    print(f"  {sample['text'][:150]}...")
    print(f"  Words: {sample['word_count']}, Priority: {sample['priority']}")

## Vocabulary Analysis

In [ ]:
# Most common words by category
vectorizer = CountVectorizer(max_features=1000, stop_words='english', min_df=5)
X_counts = vectorizer.fit_transform(df['text_clean'])
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, cat in enumerate(df['category'].unique()):
    cat_mask = df['category'] == cat
    cat_counts = np.array(X_counts[cat_mask].sum(axis=0)).flatten()
    top_idx = cat_counts.argsort()[-10:][::-1]
    
    ax = axes[idx // 2, idx % 2]
    ax.barh(range(10), cat_counts[top_idx], color='steelblue')
    ax.set_yticks(range(10))
    ax.set_yticklabels([feature_names[i] for i in top_idx])
    ax.set_title(f'Top Words: {cat}')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/nlp_01_vocabulary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/nlp_01_vocabulary.png")

## Key Findings

1. **Category Balance:** Service requests dominate (35%), followed by budget (25%) and policy (25%)
2. **Text Length:** FOIA requests are longest, service requests shortest
3. **Vocabulary:** Each category has distinct vocabulary patterns
4. **Temporal:** Steady document volume over 2-year period

Next: Baseline TF-IDF Model (Notebook 02)